# A-5b M 통합 실모델 확인 (개발 세트)

Colab **T4 GPU** 런타임에서 위에서부터 순서대로 실행한다.

- **개발 세트(`data/frozen/dev.jsonl`)만** 쓴다. 평가 세트로 프롬프트를 고치면 감점이다.
- M 파이프라인(`build_m`)은 `scripts.run_eval`이 만든다. 모델 태그·상한·예산·판정 형식(`judge.format`)은 모두 `configs/default.yaml`에서 읽는다.
- 로그는 문항마다 즉시 `results/runs/dev/r2/m.jsonl`에 쓴다. 셀이 끊겨도 같은 셀을 다시 실행하면 이어 돈다.
- 라운드는 **`r2`**를 쓴다. `r1`에는 B-4의 P1 개발 로그가 있고, 그 `RUN_META.json`의 설정 스냅샷에는 `judge` 섹션이 없어 이어 돌 수 없다.
- 실행 전에 로컬의 `YongMin` 브랜치가 GitHub에 push되어 있어야 한다.

| 볼 지표 | 기대 |
|---|---|
| 완주·오류 | 60문항 모두 기록, `error`·`timeout` 0 |
| 호출 수 | 문항당 4 이하, `budget_exhausted` 0 |
| 판정 잘림·파싱 실패 | 0에 가깝게 (C-4: 0%) |
| 정확도·호출 수·시간 | P1(`results/runs/dev/r1/p1.jsonl`)과 비교 |
| 기권률 | 답 있는 문항에서 과잉 기권이 크지 않은가 (C-4가 지적한 문제) |


## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 1. 코드 받기 (`YongMin` 브랜치)

In [ ]:
import os
os.chdir("/content")   # 다시 실행해도 repo/repo가 생기지 않게 항상 같은 위치에서 시작
REPO_URL = "https://github.com/minyi-k03/Intelligence-Software-Project.git"
# 비공개 저장소면 Colab 비밀(userdata)에 GH_TOKEN을 넣고 아래 두 줄 주석을 푼다. 토큰을 노트북에 적지 말 것.
# from google.colab import userdata
# REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")

if not os.path.isdir("repo"):
    !git clone {REPO_URL} repo
%cd /content/repo
!git fetch origin
!git checkout YongMin
!git pull --ff-only origin YongMin
!git log --oneline -3
assert os.path.isfile("tests/test_m_dev_smoke.py"), "A-5b 커밋이 없다. YongMin 브랜치가 push됐는지 확인할 것"


## 2. 의존성

In [ ]:
!pip install -q -r requirements.txt && pip list 2>/dev/null | grep -iE "^(langchain|langgraph|langchain-ollama) "

## 3. Ollama 설치와 서버 실행

In [ ]:
# lspci/lshw가 없으면 설치 스크립트가 GPU를 못 찾는다 (Colab 이미지에는 기본으로 없음) -> 먼저 설치
!apt-get install -y -q zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
!pkill -f "ollama serve" || true      # 이전 실행에서 남은 서버가 있으면 끈다
time.sleep(2)

LOG = "/content/ollama.log"           # 서버 로그를 버리지 않고 남긴다
server = subprocess.Popen(["ollama", "serve"], stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)   # 셀을 중단(Ctrl-C)해도 서버가 같이 죽지 않게 한다
for _ in range(60):                   # 응답할 때까지 최대 60초 기다린다
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("서버 준비됨")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"Ollama 서버가 60초 안에 뜨지 않았다. 서버 로그 끝: {open(LOG).read()[-2000:]}")
!ollama --version
print("--- 서버 로그에서 GPU 관련 줄")
!grep -iE "cuda|gpu|inference compute|library" {LOG} | head -10

## 4. 모델 받기 (태그는 `configs/default.yaml`에서 읽는다)

In [ ]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("model.tag =", tag)
r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)   # 진행 막대는 버리고 결과만 본다
print("pull 종료 코드:", r.returncode, "|", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 4-1. 모델이 GPU로 올라갔는지 확인
`PROCESSOR`가 `100% GPU`여야 한다. `CPU`가 보이면 멈추고 확인한다 (지연 시간이 크게 늘어난다).

In [ ]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print(subprocess.run(["ollama", "run", tag, "Say OK."], capture_output=True, text=True, timeout=300).stdout)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
assert "GPU" in ps and "CPU" not in ps.replace("PROCESSOR", ""), "모델이 GPU로 올라가지 않았다. 위 서버 로그와 ollama ps를 확인할 것"
print("OK: GPU로 로드됨")

## 5. 단위 테스트 (GPU 없이 도는 테스트)

가짜 모델 스모크(`test_m_dev_smoke.py`)가 실제 코퍼스·개발 세트 60문항으로 로그 규격을 이미 확인한다. 여기서는 환경이 같은지만 본다.

In [ ]:
!python -m pytest tests/test_graph.py tests/test_judge.py tests/test_requery.py tests/test_config.py tests/test_m_dev_smoke.py -q 2>&1 | tail -3

## 6. 동결 해시 확인
해시가 `data/README.md` 표와 다르면 여기서 멈춘다 (`run_eval`도 시작 전에 다시 확인한다).

In [ ]:
import json
from collections import Counter
from pathlib import Path

from src.eval.logger import verify_frozen_hashes

hashes = verify_frozen_hashes(Path("data/frozen"), Path("data/README.md"))
print({name: digest[:12] for name, digest in hashes.items()})
items = [json.loads(line) for line in Path("data/frozen/dev.jsonl").open(encoding="utf-8")]
print(len(items), "문항", Counter(item["type"] for item in items))
!grep -n "format" configs/default.yaml


## 7. 검증 모드 (앞 3문항, `_scratch` 묶음)

실모델이 한 문항이라도 끝까지 도는지, 로그가 규격대로 나오는지 먼저 본다. 이 결과는 git 제외 폴더에 쓴다.

In [ ]:
!python -m scripts.run_eval --verify 3 --pipelines m
rows = [json.loads(l) for l in Path("results/runs/_scratch/r1/m.jsonl").open(encoding="utf-8")]
for r in rows:
    print(r["qid"], "|", r["route"], r["decisions"], "| 답:", repr(r["answer"]), "| 정답:", r["gold"], "| calls:", r["n_calls"], "| error:", r["error"])
print(sorted(rows[0]))


## 8. 개발 세트 60문항 전체 실행

문항마다 즉시 저장한다. 끊기면 같은 셀을 다시 실행하면 이어 돈다. 설정을 바꿨다면 `r3`처럼 새 라운드로 돌려야 한다 (한 라운드에 조건이 섞이면 러너가 멈춘다).

In [ ]:
!python -m scripts.run_eval --bundle dev --round r2 --pipelines m

## 9. 요약표

집계 규칙은 `EVALUATION §4`·`INTERFACES §7.2`를 따른다.

- **정확도·기권·잘림·파싱 지표**: 워밍업 문항을 **포함한 60문항 전체**
- **비용·시간 지표**(`n_calls`, `out_tokens`, `latency_s`): 파이프라인의 첫 1문항(워밍업)을 **제외한 59문항**

| 열 | 뜻 |
|---|---|
| `retries` | 잘림 재시도 횟수 합 |
| `judge_trunc_first` | 판정 호출의 **1차 응답**이 출력 상한에 걸린 문항 비율 (재시도로 복구된 것 포함) |
| `judge_trunc` | **재시도 뒤에도** 잘린 판정 호출이 있는 문항 비율 (`scripts/compare_judge_format.py`의 `truncated_after_retry`와 같은 기준) |
| `parse_fail` | 판정 파싱 실패가 있는 문항 비율 |
| `over_abstain` | 답이 있는 문항(`answer_kind != unanswerable`) 중 기권한 비율 |


In [ ]:
import pandas as pd

OUT = Path("results/runs/dev/r2/m.jsonl")
df = pd.DataFrame([json.loads(l) for l in OUT.open(encoding="utf-8")])
df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
judge_calls = df["calls"].map(lambda cs: [c for c in cs if c["step"] == "judge"])
df["judge_trunc_first"] = judge_calls.map(lambda cs: any(c["truncated"] and not c["is_retry"] for c in cs))
df["judge_trunc"] = judge_calls.map(lambda cs: any(c["truncated"] and c["is_retry"] for c in cs))
df["parse_fail"] = judge_calls.map(lambda cs: any(c["parse_ok"] is False for c in cs))
df["total_out"] = df["calls"].map(lambda cs: sum(c["output_tokens"] for c in cs))
df["over_abstain"] = df["abstained"] & (df["answer_kind"] != "unanswerable")
cost = df[~df["is_warmup"]]   # 비용·시간 집계만 워밍업 제외. 정확도·부작용 지표는 전체(df)

print("완주:", len(df), "/ 60 | error:", int(df["error"].notna().sum()), "| timeout:", int(df["timeout"].sum()),
      "| budget_exhausted:", int(df["budget_exhausted"].sum()))
quality = df.groupby("group").agg(n=("qid", "count"), correct=("correct", "mean"), abstained=("abstained", "mean"),
                                  retries=("n_retries", "sum"), judge_trunc_first=("judge_trunc_first", "mean"),
                                  judge_trunc=("judge_trunc", "mean"), parse_fail=("parse_fail", "mean"))
spend = cost.groupby("group").agg(n_calls=("n_calls", "mean"), out_tokens=("total_out", "mean"),
                                  latency_s=("latency_s", "mean"))
pd.set_option("display.width", 200)
print(quality.join(spend).round(3))
print(f"전체 {len(df)}문항 | 정확도:", round(df["correct"].mean(), 3), "| 과잉 기권:",
      round(df["over_abstain"].sum() / (df["answer_kind"] != "unanswerable").sum(), 3))
print(f"워밍업 제외 {len(cost)}문항 | 평균 호출:", round(cost["n_calls"].mean(), 2),
      "| 평균 출력 토큰:", round(cost["total_out"].mean(), 1), "| 평균 시간:", round(cost["latency_s"].mean(), 2), "s")
print("경로:", dict(Counter(df["route"])))
print("판정:", dict(Counter(d for ds in df["decisions"] for d in ds)))
print("evidence_kept(HotpotQA):", df["evidence_kept"].dropna().mean())


## 10. P1과 비교 (같은 개발 세트, `r1`의 P1 로그)

정확도·정답 겹침은 60문항 전체, 호출 수·시간은 **양쪽 모두** 워밍업이 아닌 문항만 비교한다.


In [ ]:
P1 = Path("results/runs/dev/r1/p1.jsonl")
p1 = pd.DataFrame([json.loads(l) for l in P1.open(encoding="utf-8")])
both = df.merge(p1[["qid", "correct", "n_calls", "latency_s", "is_warmup"]], on="qid", suffixes=("_m", "_p1"))
spent = both[~both["is_warmup_m"] & ~both["is_warmup_p1"]]   # 비용·시간 비교는 워밍업 제외
print(f"문항: {len(both)} (호출·시간 비교: {len(spent)}문항, 워밍업 제외)")
print("정확도  M:", round(both["correct_m"].mean(), 3), " P1:", round(both["correct_p1"].mean(), 3))
print("호출    M:", round(spent["n_calls_m"].mean(), 2), " P1:", round(spent["n_calls_p1"].mean(), 2))
print("시간    M:", round(spent["latency_s_m"].mean(), 2), " P1:", round(spent["latency_s_p1"].mean(), 2))
print("M만 정답:", int((both["correct_m"] & ~both["correct_p1"]).sum()), " P1만 정답:", int((~both["correct_m"] & both["correct_p1"]).sum()))


## 11. 이상 문항 (원문 확인)

오류·시간 초과·판정 잘림(1차 포함)·파싱 실패·예산 소진 문항만 본다. 1차만 잘리고 재시도로 복구된 문항도 여기 나온다.


In [ ]:
odd = df[df["error"].notna() | df["timeout"] | df["judge_trunc_first"] | df["judge_trunc"] | df["parse_fail"]
         | df["budget_exhausted"]]
print(f"{len(odd)}문항")
for _, r in odd.iterrows():
    print("=" * 100)
    print(f"{r['qid']} route={r['route']} decisions={r['decisions']} answer={r['answer']!r} error={r['error']}")
    print("  calls (step, out_tokens, truncated, is_retry, parse_ok):",
          [(c["step"], c["output_tokens"], c["truncated"], c["is_retry"], c["parse_ok"]) for c in r["calls"]])


## 12. 결과 내려받기

Colab 런타임이 끝나면 파일은 사라진다. 로그(`results/runs/dev/r2/`)를 내려받아 레포에 올리고, 해석은 `docs/experiments/`에 남긴다.

In [ ]:
!cd results/runs/dev && zip -qr /content/a5b_dev_r2.zip r2
from google.colab import files
files.download("/content/a5b_dev_r2.zip")
